In [ ]:
import scanpy as sc
import numpy as np
import flowkit as fk
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Preprocessing

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from preprocessing import read_flow
bom_dir = 'BOM_CD3_01DEC25'
bom_flow, bom_samples, bom_session = read_flow(bom_dir, "BOM")

In [ ]:
lln_dir = 'LLN_CD3_01DEC25'
lln_flow, lln_samples, lln_session = read_flow(lln_dir, 'LLN')

In [ ]:
lng_dir = 'LNG_CD3_01DEC25'
lng_flow, lng_samples, lng_session = read_flow(lng_dir, 'LNG')

In [ ]:
mln_dir = 'MLN_CD3_01DEC25'
mln_flow, mln_samples, mln_session = read_flow(mln_dir, 'MLN')

In [ ]:
spl_dir = 'SPL_CD3_01DEC25'
spl_flow, spl_samples, spl_session = read_flow(spl_dir, 'SPL')

In [ ]:
df_flow = pd.concat([bom_flow, lln_flow, lng_flow, mln_flow, spl_flow])

In [ ]:
exclude = ['FSC-A', 'FSC-H', 'SSC-A', 'SSC-B-A', 'SSC-B-H',
           'SSC-H', 'AF-A', 'CD66bCD19CD326LD', 'Time', 'CD45', 'Event #']
df_flow = df_flow.drop(columns=exclude)

In [ ]:
df_flow_counts = df_flow[
    df_flow.select_dtypes(include=[np.number]).columns.difference(["age"])
]

In [ ]:
from preprocessing import pd_to_adata
adata = pd_to_adata(df_flow, df_flow_counts)

In [ ]:
adata.X = np.arcsinh(adata.X / 150)

In [ ]:
sc.pp.scale(adata, max_value=3)

In [ ]:
adata = adata[(adata[:, 'CD3'].X > 0)]
adata = adata[:, adata.var.index != 'CD3']

In [ ]:
adata.obs.rename(columns={'TCRVaJa': 'TCRva'}, inplace=True)

In [ ]:
from preprocessing import population_filter
CD4 = population_filter(adata, 'CD4', 0.0)

In [ ]:
CD8 = population_filter(adata, 'CD8', 0.0)

In [ ]:
sample = CD4
sample_name = 'CD4'
tissue_type = 'All Tissues'

In [ ]:
sc.settings.verbosity = 3

In [ ]:
plt.rcParams.update({'font.size': 10})

In [ ]:
sc.tl.pca(sample, svd_solver="arpack")
sc.pl.pca_variance_ratio(sample, log=False)

In [ ]:
sc.pl.pca_loadings(sample, components='1,2')

In [ ]:
bom_samples = bom_session.get_sample_ids()
lng_samples = lng_session.get_sample_ids()
lln_samples = lln_session.get_sample_ids()
mln_samples = mln_session.get_sample_ids()
spl_samples = spl_session.get_sample_ids()

In [ ]:
sample_list = bom_samples + lng_samples + \
    lln_samples + mln_samples + spl_samples
sample_list = list(set(sample_list))

## Clustering

In [ ]:
import harmonypy as hm
harmony_out = hm.run_harmony(
    sample.obsm['X_pca'], sample.obs, 'sample_id', max_iter_harmony=10, theta=0)
sample.obsm['X_pca_harmony'] = harmony_out.Z_corr
sc.pp.neighbors(sample, use_rep='X_pca_harmony')
sc.tl.umap(sample)

In [ ]:
markers = list(sample.var_names)

In [ ]:
sc.pl.umap(sample, color=['group'], cmap='turbo',
           title='{} {} Groups'.format(tissue_type, sample_name))

In [ ]:
sc.tl.leiden(sample, resolution=0.5, flavor='leidenalg')

In [ ]:
sample.obs['leiden'].value_counts()

In [ ]:
# sample = sample[sample.obs['leiden'] != '17', :].copy()
# sample = sample[sample.obs['leiden'] != '18', :].copy()
# sample = sample[sample.obs['leiden'] != '19', :].copy()
# sample = sample[sample.obs['leiden'] != '20', :].copy()

In [ ]:
sc.pl.umap(sample, color=['leiden'], cmap='turbo',
           title='{} {} Clusters'.format(tissue_type, sample_name))

In [ ]:
sc.pl.umap(sample, color=['tissue'], cmap='turbo',
           title='{} Tissue Groups'.format(sample_name))

In [ ]:
plt.rcParams.update({'font.size': 14})

sc.pl.umap(
    sample,
    color=markers,
    cmap='turbo',
    vmin=0,
    vmax=3,
)

In [ ]:
sc.tl.dendrogram(sample, groupby='leiden')

In [ ]:
sc.pl.dotplot(sample, markers, swap_axes=True, groupby='leiden', title="{} {} Dotplot".format(
    tissue_type, sample_name), cmap='RdBu_r', dendrogram=True, vcenter=0, vmin=-3, vmax=3)

## IL33R Expression

In [ ]:
sc.pl.violin(sample, 'IL33R', groupby='group',
             stripplot=False, order=['ctr', 'hst', 'ftl'])

In [ ]:
sc.pl.violin(sample, 'IL33R', groupby='asthma',
             stripplot=False, order=['control', 'asthmatic'])

In [ ]:
marker_genes = {
    'Tfh': ['CXCR5', 'PD-1'],
    'Th1': ['CXCR3'],
    'Th2': ['CRTH2'],
    'Trm': ['CD103', 'CD69'],
    'Treg': ['CD25', 'FOXP3'],
    'Memory': ['CCR7', 'CD45RA']
}

In [ ]:
sc.pl.dotplot(sample, marker_genes, swap_axes=True, groupby='leiden',
              cmap='RdBu_r', dendrogram=True, vcenter=0, vmin=-3, vmax=3)

In [ ]:
sc.tl.rank_genes_groups(sample, "leiden", method="t-test")

result = sample.uns["rank_genes_groups"]
groups = result["names"].dtype.names

celltype = {'celltype': []}
cluster_to_genes = {}
for group in groups:
    top_genes = result["names"][group][:3]
    cluster_to_genes[group] = f"{':'.join(top_genes)} ({group})"

celltype['celltype'] = [cluster_to_genes[leiden]
                        for leiden in sample.obs['leiden']]

In [ ]:
cell_type_series = pd.Series(celltype['celltype'])
unique_values = cell_type_series.unique()
print(unique_values)

In [ ]:
sc.pl.rank_genes_groups_dotplot(
    sample, n_genes=3,  cmap='RdBu_r', vcenter=0, vmin=-3, vmax=3)

In [ ]:
celltype = {'celltype': []}
cluster_to_genes = {
    '15': 'Effector Memory (15)',
    '16': 'Effector Memory CD103+CD69+ (16)'
    ''
}
celltype['celltype'] = [cluster_to_genes[leiden]
                        for leiden in sample.obs['leiden']]
sample.obs["celltype"] = celltype['celltype']

print(sample.obs[["leiden", "celltype"]].head())

In [ ]:
sc.tl.rank_genes_groups(sample, 'leiden', groups=[
                        '16'], reference='15', method='wilcoxon')
sc.pl.rank_genes_groups(sample, groups=['16'], n_genes=20)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
from plotting_methods import annotated_umap

annotated_umap(sample, tissue_type, sample_name, obs='celltype')

In [ ]:
for group in ['ctr', 'hst', 'ftl']:
    adata_group = sample[sample.obs['group'] == group]

    sc.pl.umap(
        adata_group,
        color=['leiden'],
        title=f'{tissue_type} {sample_name} {group} celltypes',
        cmap='turbo',
        show=False
    )

    ax = plt.gca()
    for cluster in adata_group.obs['leiden'].cat.categories:
        cluster_mask = adata_group.obs['leiden'] == cluster
        cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
        x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
        ax.text(x, y, cluster, color='black', fontsize=10,
                weight='bold', ha='center', va='center')

    plt.show()

In [ ]:
for tissue in sample.obs['tissue'].unique().tolist():
    adata_group = sample[sample.obs['tissue'] == tissue]

    sc.pl.umap(
        adata_group,
        color=['leiden'],
        title=f'{sample_name} {tissue} celltypes',
        cmap='turbo',
        show=False
    )

    ax = plt.gca()
    for cluster in adata_group.obs['leiden'].cat.categories:
        cluster_mask = adata_group.obs['leiden'] == cluster
        cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
        x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
        ax.text(x, y, cluster, color='black', fontsize=10,
                weight='bold', ha='center', va='center')

    plt.show()

In [ ]:
for tissue in sample.obs['tissue'].unique():
    for group in ['ctr', 'hst', 'ftl']:

        mask = (
            (sample.obs['tissue'] == tissue) &
            (sample.obs['group'] == group)
        )
        adata_group = sample[mask]

        if adata_group.n_obs == 0:
            continue

        sc.pl.umap(
            adata_group,
            color='celltype',
            title=f'{sample_name} {tissue} {group} celltypes',
            cmap='turbo',
            show=False
        )

        ax = plt.gca()

        for cluster in adata_group.obs['leiden'].cat.categories:
            cluster_mask = adata_group.obs['leiden'] == cluster

            if cluster_mask.sum() == 0:
                continue

            cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
            x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()

            ax.text(
                x, y, cluster,
                color='black',
                fontsize=10,
                weight='bold',
                ha='center',
                va='center'
            )

        plt.show()

In [ ]:
for group in ['ctr', 'hst', 'ftl']:
    adata_group = sample[sample.obs['group'] == group]

    sc.pl.umap(
        adata_group,
        color=['age'],
        title=f'{tissue_type} {sample_name} {group} age',
        cmap='turbo',
        show=False,
        vmin=0,
        vmax=70
    )

    ax = plt.gca()
    for cluster in adata_group.obs['leiden'].cat.categories:
        cluster_mask = adata_group.obs['leiden'] == cluster
        cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
        x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
        ax.text(x, y, cluster, color='black', fontsize=10,
                weight='bold', ha='center', va='center')

    plt.show()

In [ ]:
for tissue in sample.obs['tissue'].unique():
    for group in ['ctr', 'hst', 'ftl']:

        mask = (
            (sample.obs['tissue'] == tissue) &
            (sample.obs['group'] == group)
        )
        adata_group = sample[mask]

        if adata_group.n_obs == 0:
            continue

        sc.pl.umap(
            adata_group,
            color='age',
            title=f'{sample_name} {tissue} {group} age',
            cmap='turbo',
            show=False,
            vmin=0,
            vmax=70
        )

        ax = plt.gca()

        for cluster in adata_group.obs['leiden'].cat.categories:
            cluster_mask = adata_group.obs['leiden'] == cluster

            if cluster_mask.sum() == 0:
                continue

            cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
            x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()

            ax.text(
                x, y, cluster,
                color='black',
                fontsize=10,
                weight='bold',
                ha='center',
                va='center'
            )

        plt.show()

In [ ]:
for group in ['control', 'asthmatic',]:
    adata_group = sample[sample.obs['asthma'] == group]

    sc.pl.umap(
        adata_group,
        color=['celltype'],
        title=f'{tissue_type} {sample_name} {group} celltypes',
        cmap='turbo',
        show=False
    )

    ax = plt.gca()
    for cluster in adata_group.obs['leiden'].cat.categories:
        cluster_mask = adata_group.obs['leiden'] == cluster
        cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
        x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
        ax.text(x, y, cluster, color='black', fontsize=10,
                weight='bold', ha='center', va='center')

    plt.show()

In [ ]:
for tissue in sample.obs['tissue'].unique():
    for group in ['control', 'asthmatic']:

        mask = (
            (sample.obs['tissue'] == tissue) &
            (sample.obs['asthma'] == group)
        )
        adata_group = sample[mask]

        if adata_group.n_obs == 0:
            continue

        sc.pl.umap(
            adata_group,
            color='celltype',
            title=f'{sample_name} {tissue} {group} celltypes',
            cmap='turbo',
            show=False
        )

        ax = plt.gca()

        for cluster in adata_group.obs['leiden'].cat.categories:
            cluster_mask = adata_group.obs['leiden'] == cluster

            if cluster_mask.sum() == 0:
                continue

            cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
            x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()

            ax.text(
                x, y, cluster,
                color='black',
                fontsize=10,
                weight='bold',
                ha='center',
                va='center'
            )

        plt.show()

In [ ]:
for group in ['control', 'asthmatic',]:
    adata_group = sample[sample.obs['asthma'] == group]

    sc.pl.umap(
        adata_group,
        color=['age'],
        title=f'{tissue_type} {sample_name} {group} age',
        cmap='turbo',
        show=False
    )

    ax = plt.gca()
    for cluster in adata_group.obs['leiden'].cat.categories:
        cluster_mask = adata_group.obs['leiden'] == cluster
        cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
        x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
        ax.text(x, y, cluster, color='black', fontsize=10,
                weight='bold', ha='center', va='center')

    plt.show()

In [ ]:
for tissue in sample.obs['tissue'].unique():
    for group in ['control', 'asthmatic']:

        mask = (
            (sample.obs['tissue'] == tissue) &
            (sample.obs['asthma'] == group)
        )
        adata_group = sample[mask]

        if adata_group.n_obs == 0:
            continue

        sc.pl.umap(
            adata_group,
            color='age',
            title=f'{sample_name} {tissue} {group} age',
            cmap='turbo',
            show=False,
            vmin=0,
            vmax=70,
        )

        ax = plt.gca()

        for cluster in adata_group.obs['leiden'].cat.categories:
            cluster_mask = adata_group.obs['leiden'] == cluster

            if cluster_mask.sum() == 0:
                continue

            cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
            x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()

            ax.text(
                x, y, cluster,
                color='black',
                fontsize=10,
                weight='bold',
                ha='center',
                va='center'
            )

        plt.show()

In [ ]:
plt.rcParams.update({'font.size': 12})

In [ ]:
sc.pl.dotplot(sample, markers, swap_axes=True, groupby='leiden', title="{} {} Dotplot".format(
    tissue_type, sample_name), cmap='RdBu_r', vmin=-5, vmax=5, dendrogram=True)

In [ ]:
from plotting_methods import composition_dotplot
composition_dotplot(sample, group_x='tissue', group_y='celltype')

In [ ]:
composition_dotplot(sample, group_x='group', group_y='celltype')

In [ ]:
composition_dotplot(sample, group_x='asthma', group_y='celltype')

In [ ]:
sc.pl.stacked_violin(sample, markers, swap_axes=True, groupby='leiden', title="{} {} Violinplot".format(
    tissue_type, sample_name), cmap='RdBu_r', vmin=-5, vmax=5)